# 🚨 HMG 공급망 리스크 탐지 RAG 시스템

뉴스 기사가 입력되면 영향을 받는 협력사를 자동으로 탐지하고, 그 이유를 정리해 출력합니다.

### 데이터 흐름
```
[뉴스 입력]
    ↓ 임베딩 유사도 검색
[관련 협력사 Retrieval]  +  [관련 수입 품목 Retrieval]
    ↓ Prompt 조합
[OPEN API → 리스크 분석 보고서 생성]
```

### 사용 데이터
- `hmg_partners_900.csv` – 협력사 목록 (회사명, 업종, 비즈니스 요약)
- `import_2025.csv` – 2025년 수입 데이터 (국가, HS코드, 품목, 단가)
- `news.csv` – 뉴스 기사 (제목, 내용)


## 0. 패키지 설치

In [2]:
!pip install sentence-transformers faiss-cpu anthropic pandas numpy tqdm -q

## 1. 라이브러리 임포트 & 설정

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
# 실제 파일이 위치한 경로로 수정하세요
base_root = '/content/drive/MyDrive/aiffel_data'

In [5]:
!ls -l /content/drive/MyDrive/aiffel_data

total 471220
-rw------- 1 root root   2170705 Mar 31 07:17 2025-annual-report-ko.pdf
-rw------- 1 root root     58030 Mar 31 14:22 aluminum_alloy_import_2025.csv
-rw------- 1 root root     28836 Mar 31 06:53 catalytic_converter_import_2025.csv
-rw------- 1 root root     39853 Mar 31 14:22 cold_rolled_steel_import_2025.csv
-rw------- 1 root root     96967 Mar 31 06:43 hmg_partners_900.csv
-rw------- 1 root root    139658 Mar 31 14:22 hyundai_semiconductor_import_2025.csv
-rw------- 1 root root    882262 Mar 31 14:18 hyundai_supply_chain_news_final_v2.csv
-rw------- 1 root root    561063 Mar 31 06:52 import_2025.csv
-rw------- 1 root root  57127652 Mar 31 15:47 kotra_news_20250101_20250331.csv
-rw------- 1 root root  66629414 Mar 31 15:49 kotra_news_20250401_20250630.csv
-rw------- 1 root root 112060838 Mar 31 15:49 kotra_news_20250701_20250930.csv
-rw------- 1 root root 160845098 Mar 31 15:49 kotra_news_20251001_20251231.csv
-rw------- 1 root root  80831524 Mar 31 15:51 kotra_news_20260

In [6]:
import os
import json
import warnings
import numpy as np
import pandas as pd
import faiss
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
import anthropic
from google.colab import userdata

warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────
# ✏️  설정값: 필요에 따라 수정하세요
# ─────────────────────────────────────────────
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')

# 한국어 지원 다국어 임베딩 모델
EMBEDDING_MODEL = "paraphrase-multilingual-MiniLM-L12-v2"

# 검색 시 반환할 최대 문서 수
TOP_K_PARTNERS = 10   # 협력사 후보
TOP_K_IMPORTS  = 8    # 수입 품목 후보
TOP_K_NEWS     = 5    # 유사 뉴스 후보

# 데이터 파일 경로 (같은 디렉토리 또는 경로 수정)
PATH_PARTNERS     = os.path.join(base_root,"hmg_partners_900.csv")
PATH_IMPORTS      = os.path.join(base_root,"import_2025.csv")
PATH_NEWS         = os.path.join(base_root,"hyundai_supply_chain_news_final_v2.csv")
PATH_OVERSEA_NEWS = os.path.join(base_root,"kotra_news_combined.csv")
# PATH_PROCESS     = os.path.join(base_root,"한국산업단지공단_자동차_부품_제조업_공정_합성데이터_20250925.csv")

print("✅ 설정 완료")

✅ 설정 완료


## 2. 데이터 로드 & 전처리

In [7]:
# ── 협력사 데이터 ──────────────────────────────
df_partners = pd.read_csv(PATH_PARTNERS)
df_partners.columns = df_partners.columns.str.strip()
df_partners = df_partners.fillna("")
print(f"협력사: {len(df_partners)}개")
print(df_partners.head(3).to_string())
print()

협력사: 901개
      회사명           업종                                비즈니스 요약
0  현대트랜시스       자동차 부품  변속기, 액슬, 시트 등 자동차 핵심 파워트레인 및 내장 부품 제조
1    현대위아  자동차 부품 및 기계     엔진, 구동부품 등 자동차 부품과 공작기계 및 방산 부품 생산
2   현대케피코       자동차 부품     엔진 및 변속기 제어기, 센서류 등 차량용 전자제어시스템 제조



In [8]:
# ── 수입 데이터 ──────────────────────────────
df_imports = pd.read_csv(PATH_IMPORTS)
df_imports = df_imports.fillna(0)
print(f"수입 레코드: {len(df_imports)}개")
print(df_imports.head(3).to_string())
print()

# 국가 + 품목 기준으로 집계 (단가 평균, 수입금액 합)
df_imp_agg = (
    df_imports
    .groupby(["country_name", "hs_code", "item_name"], as_index=False)
    .agg(
        avg_unit_price=("unit_price", "mean"),
        total_value=("imp_value", "sum"),
        months=("period", "count")
    )
)
df_imp_agg = df_imp_agg[df_imp_agg["total_value"] > 0].reset_index(drop=True)
print(f"집계 후 수입 레코드: {len(df_imp_agg)}개")

수입 레코드: 5725개
    period country_code country_name     hs_code             item_name  imp_weight  imp_value  unit_price
0  2025.01           AE    아랍에미리트 연합  7209169000                    기타         0.0        0.0         0.0
1  2025.01           BD        방글라데시  7209169000                    기타         0.0        0.0         0.0
2  2025.01           BE          벨기에  7209161000  인장강도가 340메가파스칼 이상인 것         0.0        0.0         0.0

집계 후 수입 레코드: 507개


In [9]:
import pandas as pd

# ── 뉴스 데이터 로드 (인코딩 및 엔진 설정 추가) ──────────────────────────────
try:
    # 1. 한국어 윈도우 표준 인코딩(cp949)으로 시도
    df_news = pd.read_csv(PATH_NEWS, encoding='cp949', sep='\t') # Add sep='\t' for tab-separated values
except UnicodeDecodeError:
    # 2. 위 방법이 안 될 경우 다른 한국어 인코딩(euc-kr)으로 시도
    df_news = pd.read_csv(PATH_NEWS, encoding='euc-kr', sep='\t') # Add sep='\t' for tab-separated values
except Exception as e:
    # 3. 만약 데이터 구조가 복잡해 ParserError가 난다면 엔진을 python으로 변경
    df_news = pd.read_csv(PATH_NEWS, encoding='cp949', engine='python', on_bad_lines='skip', sep='\t') # Add sep='\t' for tab-separated values

# 컬럼명 공백 제거 및 전처리
df_news.columns = df_news.columns.str.strip()
df_news = df_news.fillna("")

# 결과 확인
print(f"✅ 뉴스 데이터 {len(df_news)}개 로드 완료")
display(df_news.head()) # Use display for better output in Colab

✅ 뉴스 데이터 1134개 로드 완료


,title,link,pub_date,content
0,로봇은 왜 인간의 모습을 하고 있는가?,http://www.brainmedia.co.kr/Opinion/25313,"Tue, 31 Mar 2026 08:44:00 +0900",분야별 TV 매거진 한양대 공학대학 로봇공학과 한재권 교수 한재권 교수는 한양대학교...
1,[결정의 순간들①] 정주영·정세영의 유산 'HDC 정몽규',https://www.meconomynews.com/news/articleView....,"Tue, 31 Mar 2026 09:38:00 +0900",정몽규 회장은 대한민국 재벌이다. 고 정주영 회장의 조카로 태어나 현대자동차 회장으...
2,[이슈체크] 이란전쟁 한달 만에 시총 840조 증발…'삼전닉스'만 372조원...,https://www.tfmedia.co.kr/news/article.html?no...,"Tue, 31 Mar 2026 08:50:00 +0900",2026.03.31 이란전쟁 여파로 3월 한 달간 국내 증시 상장사 전체 시가총액이...
3,"[TECH한주] 'K-라이다'의 자존심 에스오에스랩, 피지컬 AI의 '눈'을 넘어...",https://www.epnc.co.kr/news/articleView.html?i...,"Tue, 31 Mar 2026 08:00:00 +0900",에스오에스랩은 2016년 광주과학기술원 박사급 인력들이 의기투합해 설립한 국내 최고...
4,YTN '저널리즘 책무위' 가동… 노조 &quot;유진 강점기 합리화 명분&quot;,https://n.news.naver.com/mnews/article/127/000...,"Mon, 30 Mar 2026 15:25:00 +0900","30일 이사회 김백 대국민사과, 현대자동차 기사 삭제 조사YTN지부 양상우 사단 경..."


In [10]:
print(df_news.columns)

Index(['title', 'link', 'pub_date', 'content'], dtype='object')


In [11]:
print("df_news 컬럼:")
print(df_news.columns)
print("\ndf_news 정보 (컬럼 및 타입):")
df_news.info()

df_news 컬럼:
Index(['title', 'link', 'pub_date', 'content'], dtype='object')

df_news 정보 (컬럼 및 타입):
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1134 entries, 0 to 1133
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   title     1134 non-null   object
 1   link      1134 non-null   object
 2   pub_date  1134 non-null   object
 3   content   1134 non-null   object
dtypes: object(4)
memory usage: 35.6+ KB


In [12]:
import pandas as pd

# ── 뉴스 데이터 로드 (인코딩 및 엔진 설정 추가) ──────────────────────────────
df_oversea_news = pd.read_csv(PATH_OVERSEA_NEWS)
# try:
#     # 1. 한국어 윈도우 표준 인코딩(cp949)으로 시도
#     df_oversea_news = pd.read_csv(PATH_OVERSEA_NEWS, encoding='cp949', sep='\t') # Add sep='\t' for tab-separated values
# except UnicodeDecodeError:
#     # 2. 위 방법이 안 될 경우 다른 한국어 인코딩(euc-kr)으로 시도
#     df_oversea_news = pd.read_csv(PATH_OVERSEA_NEWS, encoding='euc-kr', sep='\t') # Add sep='\t' for tab-separated values
# except Exception as e:
#     # 3. 만약 데이터 구조가 복잡해 ParserError가 난다면 엔진을 python으로 변경
#     df_oversea_news = pd.read_csv(PATH_OVERSEA_NEWS, encoding='cp949', engine='python', on_bad_lines='skip', sep='\t') # Add sep='\t' for tab-separated values

# 컬럼명 공백 제거 및 전처리
df_oversea_news.columns = df_oversea_news.columns.str.strip()
df_oversea_news = df_oversea_news.fillna("")
tmp_df = df_oversea_news[['newsTitl', 'kotraNewsUrl', 'othbcDt', 'cntntSumar']]
df_oversea_news = tmp_df.rename(columns={
    'newsTitl': 'title',
    'kotraNewsUrl': 'link',
    'othbcDt': 'pub_date',
    'cntntSumar': 'content'
})


# 결과 확인
print(f"✅ 뉴스 데이터 {len(df_oversea_news)}개 로드 완료")
display(df_oversea_news.head()) # Use display for better output in Colab

✅ 뉴스 데이터 317개 로드 완료


,title,link,pub_date,content
0,공장 넘어 도시로&hellip; 일상으로 확산하는 중국 &lsquo;피지컬 AI&r...,https://dream.kotra.or.kr/user/extra/kotranews...,2026-03-31,"중국 로봇 산업은 제조업을 넘어 전력 인프라, 렌탈 서비스, 돌봄, 리테일 등 실제..."
1,"﻿페루 광업의 기회와 과제, 중남미 광업&middot;환경 국제 콘퍼런스와 인터뷰로...",https://dream.kotra.or.kr/user/extra/kotranews...,2026-03-31,"페루는 풍부한 광물자원과 신규 투자 잠재력을 보유하나, 정치 불안정&middot;인..."
2,2025년 중국 조선산업 결산 이모저모,https://dream.kotra.or.kr/user/extra/kotranews...,2026-03-31,"2025년 글로벌 조선 산업은 &#39;안정 속 조정, 친환경 전환&#39;의 중요..."
3,"[중국 톈진 대해부 시리즈①] 中 북부의 소비 거점, 톈진시 소비자는 무엇에 지갑을...",https://dream.kotra.or.kr/user/extra/kotranews...,2026-03-31,중국 4대 직할시 중 하나인 톈진시는 북방 최대 규모의 종합항만을 보유한 교통&mi...
4,"브라질 일자리 대이동 시대 &mdash; 북동부 고용난 vs 남부 인력난, AI&m...",https://dream.kotra.or.kr/user/extra/kotranews...,2026-03-31,브라질 고용시장은 평균 지표와 달리 지역&middot;세대&middot;성별별 격차...


In [13]:
# df_news와 df_oversea_news를 결합
df_news = pd.concat([df_news, df_oversea_news], ignore_index=True)

print(f"✅ 결합된 뉴스 데이터: {len(df_news)}개")
display(df_news)

✅ 결합된 뉴스 데이터: 1451개


,title,link,pub_date,content
0,로봇은 왜 인간의 모습을 하고 있는가?,http://www.brainmedia.co.kr/Opinion/25313,"Tue, 31 Mar 2026 08:44:00 +0900",분야별 TV 매거진 한양대 공학대학 로봇공학과 한재권 교수 한재권 교수는 한양대학교...
1,[결정의 순간들①] 정주영·정세영의 유산 'HDC 정몽규',https://www.meconomynews.com/news/articleView....,"Tue, 31 Mar 2026 09:38:00 +0900",정몽규 회장은 대한민국 재벌이다. 고 정주영 회장의 조카로 태어나 현대자동차 회장으...
2,[이슈체크] 이란전쟁 한달 만에 시총 840조 증발…'삼전닉스'만 372조원...,https://www.tfmedia.co.kr/news/article.html?no...,"Tue, 31 Mar 2026 08:50:00 +0900",2026.03.31 이란전쟁 여파로 3월 한 달간 국내 증시 상장사 전체 시가총액이...
3,"[TECH한주] 'K-라이다'의 자존심 에스오에스랩, 피지컬 AI의 '눈'을 넘어...",https://www.epnc.co.kr/news/articleView.html?i...,"Tue, 31 Mar 2026 08:00:00 +0900",에스오에스랩은 2016년 광주과학기술원 박사급 인력들이 의기투합해 설립한 국내 최고...
4,YTN '저널리즘 책무위' 가동… 노조 &quot;유진 강점기 합리화 명분&quot;,https://n.news.naver.com/mnews/article/127/000...,"Mon, 30 Mar 2026 15:25:00 +0900","30일 이사회 김백 대국민사과, 현대자동차 기사 삭제 조사YTN지부 양상우 사단 경..."
...,...,...,...,...
1446,"대만 디저트 시장의 변화, 일본발 &lsquo;생도넛&rsquo; 확산",https://dream.kotra.or.kr/user/extra/kotranews...,2026-01-06,최근 대만에서 일본발 &lsquo;생도넛&rsquo;이 기존 도넛을 대체하며 핵심 ...
1447,파라과이중앙은행(BCP)의 2025년 파라과이 경제분석 및 2026년 전망,https://dream.kotra.or.kr/user/extra/kotranews...,2026-01-02,파라과이 경제는 2025년 약 6%의 높은 성장률을 기록하며 중남미 평균을 웃도는 ...
1448,2026년 독일의 주요 제도 변화 한눈에 보기,https://dream.kotra.or.kr/user/extra/kotranews...,2026-01-05,2026년을 전후해 독일&middot;EU에서는 CBAM 본격 시행을 비롯해 PPW...
1449,"독일 배터리이행법 시행, 판매 중단 우려 완화 속 규제 대응이 경쟁력으로 부상",https://dream.kotra.or.kr/user/extra/kotranews...,2026-01-05,독일은 EU 배터리 규정을 이행하며 배터리이행법(BattDG)을 시행했다. 지난 2...


In [14]:
import pandas as pd

# 예시 데이터프레임의 'pub_date' 컬럼을 변환
# df_news의 'pub_date' 컬럼을 datetime 객체로 변환
df_news['pub_date'] = pd.to_datetime(df_news['pub_date'], errors='coerce')

# 'YYYY-MM-DD' 형식의 문자열로 포맷팅
df_news['pub_date'] = df_news['pub_date'].dt.strftime('%Y-%m-%d')

display(df_news)

,title,link,pub_date,content
0,로봇은 왜 인간의 모습을 하고 있는가?,http://www.brainmedia.co.kr/Opinion/25313,2026-03-31,분야별 TV 매거진 한양대 공학대학 로봇공학과 한재권 교수 한재권 교수는 한양대학교...
1,[결정의 순간들①] 정주영·정세영의 유산 'HDC 정몽규',https://www.meconomynews.com/news/articleView....,2026-03-31,정몽규 회장은 대한민국 재벌이다. 고 정주영 회장의 조카로 태어나 현대자동차 회장으...
2,[이슈체크] 이란전쟁 한달 만에 시총 840조 증발…'삼전닉스'만 372조원...,https://www.tfmedia.co.kr/news/article.html?no...,2026-03-31,2026.03.31 이란전쟁 여파로 3월 한 달간 국내 증시 상장사 전체 시가총액이...
3,"[TECH한주] 'K-라이다'의 자존심 에스오에스랩, 피지컬 AI의 '눈'을 넘어...",https://www.epnc.co.kr/news/articleView.html?i...,2026-03-31,에스오에스랩은 2016년 광주과학기술원 박사급 인력들이 의기투합해 설립한 국내 최고...
4,YTN '저널리즘 책무위' 가동… 노조 &quot;유진 강점기 합리화 명분&quot;,https://n.news.naver.com/mnews/article/127/000...,2026-03-30,"30일 이사회 김백 대국민사과, 현대자동차 기사 삭제 조사YTN지부 양상우 사단 경..."
...,...,...,...,...
1446,"대만 디저트 시장의 변화, 일본발 &lsquo;생도넛&rsquo; 확산",https://dream.kotra.or.kr/user/extra/kotranews...,NaN,최근 대만에서 일본발 &lsquo;생도넛&rsquo;이 기존 도넛을 대체하며 핵심 ...
1447,파라과이중앙은행(BCP)의 2025년 파라과이 경제분석 및 2026년 전망,https://dream.kotra.or.kr/user/extra/kotranews...,NaN,파라과이 경제는 2025년 약 6%의 높은 성장률을 기록하며 중남미 평균을 웃도는 ...
1448,2026년 독일의 주요 제도 변화 한눈에 보기,https://dream.kotra.or.kr/user/extra/kotranews...,NaN,2026년을 전후해 독일&middot;EU에서는 CBAM 본격 시행을 비롯해 PPW...
1449,"독일 배터리이행법 시행, 판매 중단 우려 완화 속 규제 대응이 경쟁력으로 부상",https://dream.kotra.or.kr/user/extra/kotranews...,NaN,독일은 EU 배터리 규정을 이행하며 배터리이행법(BattDG)을 시행했다. 지난 2...


In [15]:
# # ── 한국산업단지공단_자동차_부품_제조업_공정_합성데이터_20250925 ──────────────────────────────
# # 기존 코드에서 encoding 옵션을 추가합니다.
# try:
#     df_process = pd.read_csv(PATH_PROCESS, encoding='cp949')
# except UnicodeDecodeError:
#     # 만약 cp949로도 안 된다면 euc-kr을 시도합니다.
#     df_process = pd.read_csv(PATH_PROCESS, encoding='euc-kr')

# # 데이터 전처리
# df_process = df_process.fillna("")

# # 중복 제거 (공장번호와 공정명이 같은 경우 제거)
# if "공장관리번호" in df_process.columns and "공정명" in df_process.columns:
#     df_process = df_process.drop_duplicates(subset=["공장관리번호", "공정명"]).reset_index(drop=True)

# print(f"✅ 자동차 부품 공정 데이터: {len(df_process)}개 로드 완료")
# print(df_process[["공정명", "공정도명", "공정설명"]].head(5).to_string())

## 3. 문서(Document) 코퍼스 구성

각 데이터를 임베딩 가능한 텍스트 문서로 변환합니다.

In [16]:
def build_partner_doc(row):
    """협력사 → 임베딩용 텍스트"""
    return (
        f"회사명: {row['회사명']}\n"
        f"업종: {row['업종']}\n"
        f"비즈니스 요약: {row['비즈니스 요약']}"
    )

def build_import_doc(row):
    """수입 레코드 → 임베딩용 텍스트"""
    return (
        f"수입국: {row['country_name']}\n"
        f"HS코드: {row['hs_code']}\n"
        f"품목: {row['item_name']}\n"
        f"평균단가: {row['avg_unit_price']:.2f} USD/kg\n"
        f"총수입액: {row['total_value']:.0f} 천달러"
    )

def build_news_doc(row):
    """뉴스 기사 → 임베딩용 텍스트 (제목 + 본문 앞 500자)"""
    content_preview = str(row.get("content", ""))[:500]
    return f"제목: {row['title']}\n내용: {content_preview}"

# 문서 리스트 생성
partner_docs  = [build_partner_doc(r) for _, r in df_partners.iterrows()]
import_docs   = [build_import_doc(r) for _, r in df_imp_agg.iterrows()]
news_docs     = [build_news_doc(r)   for _, r in df_news.iterrows()]

print(f"협력사 문서: {len(partner_docs)}개")
print(f"수입 문서:   {len(import_docs)}개")
print(f"뉴스 문서:   {len(news_docs)}개")
print()
print("[협력사 문서 예시]")
print(partner_docs[0])
print()
print("[수입 문서 예시]")
print(import_docs[0])

협력사 문서: 901개
수입 문서:   507개
뉴스 문서:   1451개

[협력사 문서 예시]
회사명: 현대트랜시스
업종: 자동차 부품
비즈니스 요약: 변속기, 액슬, 시트 등 자동차 핵심 파워트레인 및 내장 부품 제조

[수입 문서 예시]
수입국: 가봉
HS코드: 7601209000
품목: 기타
평균단가: 2.33 USD/kg
총수입액: 476534 천달러


## 4. 임베딩 모델 로드 & FAISS 인덱스 구축

In [17]:
print(f"임베딩 모델 로딩: {EMBEDDING_MODEL}")
embedder = SentenceTransformer(EMBEDDING_MODEL)
print("✅ 모델 로드 완료")

임베딩 모델 로딩: paraphrase-multilingual-MiniLM-L12-v2


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ 모델 로드 완료


In [18]:
def build_faiss_index(docs, model, batch_size=64):
    """문서 리스트 → FAISS L2 인덱스 반환"""
    print(f"  임베딩 생성 중 ({len(docs)}개 문서)...")
    embeddings = model.encode(
        docs,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True   # cosine similarity 사용 위해 정규화
    )
    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)  # Inner Product = cosine (정규화된 벡터)
    index.add(embeddings)
    print(f"  ✅ FAISS 인덱스 완료 (dim={dim}, 문서수={index.ntotal})")
    return index, embeddings

print("[1/3] 협력사 인덱스 구축...")
partner_index, partner_embeddings = build_faiss_index(partner_docs, embedder)

print("[2/3] 수입품목 인덱스 구축...")
import_index, import_embeddings = build_faiss_index(import_docs, embedder)

print("[3/3] 뉴스 인덱스 구축...")
news_index, news_embeddings = build_faiss_index(news_docs, embedder)

print("\n✅ 모든 FAISS 인덱스 구축 완료")

[1/3] 협력사 인덱스 구축...
  임베딩 생성 중 (901개 문서)...


Batches:   0%|          | 0/15 [00:00<?, ?it/s]

  ✅ FAISS 인덱스 완료 (dim=384, 문서수=901)
[2/3] 수입품목 인덱스 구축...
  임베딩 생성 중 (507개 문서)...


Batches:   0%|          | 0/8 [00:00<?, ?it/s]

  ✅ FAISS 인덱스 완료 (dim=384, 문서수=507)
[3/3] 뉴스 인덱스 구축...
  임베딩 생성 중 (1451개 문서)...


Batches:   0%|          | 0/23 [00:00<?, ?it/s]

  ✅ FAISS 인덱스 완료 (dim=384, 문서수=1451)

✅ 모든 FAISS 인덱스 구축 완료


## 5. 검색(Retrieval) 함수 정의

In [19]:
def retrieve(query: str, index, docs: list, metadata_df: pd.DataFrame, top_k: int = 5):
    """
    쿼리 → FAISS 검색 → (score, doc_text, metadata_row) 리스트 반환
    """
    q_emb = embedder.encode(
        [query],
        normalize_embeddings=True,
        convert_to_numpy=True
    )
    scores, indices = index.search(q_emb, top_k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx >= 0:
            results.append({
                "score": float(score),
                "doc": docs[idx],
                "meta": metadata_df.iloc[idx].to_dict()
            })
    return results


def retrieve_by_news(news_title: str, news_content: str = ""):
    """
    뉴스 제목+내용 → 관련 협력사 & 수입품목 검색
    """
    query = f"{news_title} {news_content[:300]}".strip()

    partners = retrieve(query, partner_index, partner_docs, df_partners, TOP_K_PARTNERS)
    imports  = retrieve(query, import_index,  import_docs,  df_imp_agg,  TOP_K_IMPORTS)

    return partners, imports


print("✅ 검색 함수 정의 완료")

# 동작 확인
test_q = "이란 전쟁으로 인한 유가 상승"
test_partners, test_imports = retrieve_by_news(test_q)
print(f"\n[테스트] 쿼리: '{test_q}'")
print(f"  검색된 협력사: {len(test_partners)}개 (상위 유사도: {test_partners[0]['score']:.3f})")
print(f"  검색된 수입품목: {len(test_imports)}개 (상위 유사도: {test_imports[0]['score']:.3f})")

✅ 검색 함수 정의 완료

[테스트] 쿼리: '이란 전쟁으로 인한 유가 상승'
  검색된 협력사: 10개 (상위 유사도: 0.318)
  검색된 수입품목: 8개 (상위 유사도: 0.346)


## 6. OPEN API 기반 리스크 분석 함수

In [20]:
!pip install -q openai

import openai
from google.colab import userdata

client = openai.OpenAI(api_key=OPENAI_API_KEY)

SYSTEM_PROMPT = """
당신은 현대자동차그룹(HMG) 구매/공급망 전문 리스크 분석 AI입니다.
주어진 뉴스와 검색된 협력사·수입 품목 데이터를 바탕으로,
어떤 협력사가 영향을 받는지 판단하고 그 이유를 체계적으로 정리합니다.

분석 시 다음 원칙을 따릅니다:
1. 뉴스에서 핵심 리스크 요인(지정학적 이슈, 원자재 가격, 공급 차질 등)을 식별합니다.
2. 검색된 협력사 중 실제로 영향을 받을 가능성이 높은 업체를 선별합니다.
3. 수입 품목 데이터와 연계하여 원자재/소재 공급 리스크를 분석합니다.
4. 리스크 수준을 [높음/중간/낮음]으로 명확히 표시합니다.
5. 권고 조치(선제적 단가 협상, 대체 공급선 검토 등)를 제안합니다.
6. 근거가 부족한 연결고리는 추측임을 명시합니다.
"""

def format_retrieved_context(
    partners: list,
    imports: list,
    partner_score_threshold: float = 0.3,
    import_score_threshold: float = 0.3
) -> str:
    """검색 결과를 프롬프트용 컨텍스트 문자열로 변환"""

    # 임계값 이상 협력사만 포함
    filtered_partners = [p for p in partners if p["score"] >= partner_score_threshold]
    filtered_imports  = [i for i in imports  if i["score"] >= import_score_threshold]

    ctx = "[검색된 협력사 후보]\n"
    if filtered_partners:
        for i, p in enumerate(filtered_partners, 1):
            ctx += f"\n{i}. (유사도: {p['score']:.3f})\n{p['doc']}\n"
    else:
        ctx += "(유사도 기준 이상의 협력사 없음)\n"

    ctx += "\n[검색된 수입 품목 현황]\n"
    if filtered_imports:
        for i, imp in enumerate(filtered_imports, 1):
            ctx += f"\n{i}. (유사도: {imp['score']:.3f})\n{imp['doc']}\n"
    else:
        ctx += "(유사도 기준 이상의 수입 품목 없음)\n"

    return ctx


def analyze_risk(
    news_title: str,
    news_content: str = "",
    news_date: str = "",
    verbose: bool = True
) -> str:
    """
    뉴스 입력 → 리스크 분석 보고서 반환

    Parameters
    ----------
    news_title   : 뉴스 제목
    news_content : 뉴스 본문 (없으면 제목만 사용)
    news_date    : 발행일 (참고용)
    verbose      : 중간 과정 출력 여부

    Returns
    -------
    str : 리스크 분석 보고서 (마크다운 형식)
    """
    if verbose:
        print(f"🔍 뉴스 분석 시작: {news_title[:60]}...")

    # ── Step 1: Retrieval ──────────────────────
    partners, imports = retrieve_by_news(news_title, news_content)
    context = format_retrieved_context(partners, imports)

    if verbose:
        print(f"   협력사 후보 {len(partners)}개, 수입품목 {len(imports)}개 검색 완료")

    # ── Step 2: Prompt 구성 ────────────────────
    content_preview = news_content[:800] if news_content else "(본문 없음)"

    user_prompt = f"""다음 뉴스를 분석하여 영향 받는 HMG 협력사와 리스크 원인을 보고서 형식으로 작성해주세요.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📰 뉴스 정보
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
발행일: {news_date if news_date else '미상'}
제목: {news_title}
본문 (요약):
{content_preview}

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📦 RAG 검색 결과 (협력사 & 수입품목 데이터)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
{context}

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📋 작성 요청
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
아래 형식으로 보고서를 작성해주세요:

## 1. 뉴스 핵심 리스크 요인
(뉴스에서 공급망에 영향을 줄 수 있는 핵심 요소 3~5가지)

## 2. 영향 받는 협력사 분석
(각 협력사별로: 회사명, 리스크 수준[높음/중간/낮음], 영향 받는 이유)
단, 임베딩 유사도가 높더라도 실제 연관성이 낮다면 제외하세요.

## 3. 수입 원자재/품목 리스크
(어떤 수입품목/국가가 영향을 받는지, 단가 변동 예상)

## 4. 공급망 연결 시나리오
(뉴스 → 수입품목 영향 → 협력사 영향 → 최종 생산 차질로 이어지는 인과관계 서술)

## 5. 권고 조치
(단기/중기 대응 방안, 우선순위 포함)

## 6. 분석 한계 및 주의사항
(데이터 부족이나 추측에 의한 부분 명시)
"""

    # ── Step 3: OPEN API 호출 ────────────────
    if verbose:
        print("   OPEN API 호출 중...")

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            # 1. 시스템 역할 부여 (앞서 정의한 SYSTEM_PROMPT 사용)
            {"role": "system", "content": SYSTEM_PROMPT},
            # 2. 상세 분석 요청 (앞서 정의한 user_prompt 사용)
            {"role": "user", "content": user_prompt}
        ],
        max_tokens=3000,
        temperature=0  # 일관성 있는 분석을 위해 0 유지
    )

    report = response.choices[0].message.content
    return report # 분석 결과(보고서)를 반환하도록 return 추가

print("✅ 분석 함수 정의 완료")

✅ 분석 함수 정의 완료


## 7. 단일 뉴스 분석 실행

In [21]:
# ── 분석할 뉴스 선택 (news.csv에서 가져오기) ──────────
# 직접 선택하거나 아래처럼 특정 인덱스 사용
sample_idx = 0  # ← 0, 1, 2 ... 변경해서 다른 뉴스 분석 가능

sample_news = df_news.iloc[sample_idx]
print(f"선택된 뉴스:\n제목: {sample_news['title']}")
print(f"날짜: {sample_news.get('pub_date', '미상')}")
print(f"본문(앞 200자): {str(sample_news.get('content', ''))[:200]}...")

선택된 뉴스:
제목: 로봇은 왜 인간의 모습을 하고 있는가?
날짜: 2026-03-31
본문(앞 200자): 분야별 TV 매거진 한양대 공학대학 로봇공학과 한재권 교수 한재권 교수는 한양대학교 로봇공학과 부교수이자, 휴머노이드 전문 스타트업 AeiROBOT 의 최고기술책임자로 활동하며, 실제 산업 현장 적용이 가능한 로봇 개발을 선도하고 있습니다. 세계적 로봇대회 RoboCup Humanoid League 우승 경력을 바탕으로, 사람처럼 걷고 조작할 수 있는 휴머...


In [22]:
# ── 리스크 분석 실행 ──────────────────────────
# 2. 함수를 호출하여 'report' 변수에 결과 저장
# (이 과정에서 "OPEN API 호출 중..." 메시지가 뜰 거예요)
report = analyze_risk(
    news_title   = sample_news["title"],
    news_content = sample_news.get("content", ""),
    news_date    = sample_news.get("pub_date", "")
)

# 3. 결과 출력
print("\n📊 공급망 리스크 분석 보고서")
print("=" * 70)
print(report)

🔍 뉴스 분석 시작: 로봇은 왜 인간의 모습을 하고 있는가?...
   협력사 후보 10개, 수입품목 8개 검색 완료
   OPEN API 호출 중...

📊 공급망 리스크 분석 보고서
## 1. 뉴스 핵심 리스크 요인
1. **로봇 기술 발전**: 휴머노이드 로봇 및 AI 기술의 발전이 산업 전반에 영향을 미칠 것으로 예상됨.
2. **경쟁 심화**: 글로벌 기업들이 로봇 및 AI 분야에 적극적으로 투자하고 있어 경쟁이 치열해짐.
3. **산업용 로봇 수요 증가**: 인구 절벽 문제 해결을 위한 산업용 로봇의 필요성이 증가하고 있음.
4. **기술 혁신 속도**: 기술 혁신의 속도가 빨라져 기존 협력사들이 적시에 대응하지 못할 위험이 존재함.

## 2. 영향 받는 협력사 분석
1. **회사명**: 레인보우로보틱스  
   **리스크 수준**: 높음  
   **영향 받는 이유**: 인간형 로봇 및 산업용 협동 로봇 기술 개발에 집중하고 있어, 경쟁 심화와 기술 혁신 속도 증가로 인해 시장 점유율 감소 위험이 있음.

2. **회사명**: 로보케어  
   **리스크 수준**: 중간  
   **영향 받는 이유**: 사회 약자를 위한 돌봄 로봇을 생산하고 있으나, 산업용 로봇 시장의 변화에 따라 기술적 대응이 필요할 수 있음.

3. **회사명**: 유일로보틱스  
   **리스크 수준**: 중간  
   **영향 받는 이유**: 산업용 로봇 및 자동화 솔루션 제공업체로, 로봇 기술 발전에 따른 경쟁 심화로 인해 시장에서의 위치가 위협받을 수 있음.

4. **회사명**: 로보스타  
   **리스크 수준**: 중간  
   **영향 받는 이유**: 자동차 제조 공정용 로봇을 공급하고 있어, 산업용 로봇 수요 증가에 따른 기회와 함께 경쟁 심화로 인한 리스크가 존재함.

## 3. 수입 원자재/품목 리스크
현재 검색된 수입 품목 데이터가 없으므로, 특정 수입 품목이나 국가에 대한 리스크 분석이 불가능함. 그러나 로봇 부품 및 기술 관련 수입 품목이 영향을 받

## 8. 사용자 정의 뉴스 입력 분석

In [23]:
# ── 직접 뉴스 텍스트를 입력해서 분석 ────────────
custom_title = "중동 분쟁 장기화로 알루미늄·철강 원자재 가격 30% 급등"
custom_content = """
미국-이란 갈등이 심화되면서 중동발 원자재 공급 불안이 가속화되고 있다.
호르무즈 해협을 통한 원유 및 알루미늄 원자재 수송이 차질을 빚고 있으며,
런던금속거래소(LME)에서 알루미늄 현물가가 전월 대비 28% 급등했다.
철강 슬래브와 캐스팅 얼로이 가격도 동반 상승세를 보이고 있어
자동차 부품 제조업체들의 원가 부담이 크게 증가할 것으로 예상된다.
특히 단조, 주조, 금속 가공 업종에 직격탄이 예상된다.
"""

custom_report = analyze_risk(
    news_title=custom_title,
    news_content=custom_content,
    verbose=True
)

print("=" * 70)
print("📊 공급망 리스크 분석 보고서 (사용자 정의 뉴스)")
print("=" * 70)
print(custom_report)

🔍 뉴스 분석 시작: 중동 분쟁 장기화로 알루미늄·철강 원자재 가격 30% 급등...
   협력사 후보 10개, 수입품목 8개 검색 완료
   OPEN API 호출 중...
📊 공급망 리스크 분석 보고서 (사용자 정의 뉴스)
## 1. 뉴스 핵심 리스크 요인
1. 중동 지역의 지정학적 갈등 심화 (미국-이란 갈등)
2. 호르무즈 해협을 통한 원자재 수송 차질
3. 알루미늄 및 철강 원자재 가격 급등 (28% 이상)
4. 자동차 부품 제조업체의 원가 부담 증가
5. 단조, 주조, 금속 가공 업종에 대한 직접적인 영향

## 2. 영향 받는 협력사 분석
1. **피제이메탈**
   - 리스크 수준: 높음
   - 영향 받는 이유: 알루미늄 제련업체로서 원자재 가격 상승이 직접적인 원가 부담으로 작용할 가능성이 높음.

2. **세진메탈**
   - 리스크 수준: 높음
   - 영향 받는 이유: 자동차용 알루미늄 합금 및 잉곳 생산업체로, 알루미늄 가격 상승이 생산 비용에 큰 영향을 미칠 것으로 예상됨.

3. **남선알미늄**
   - 리스크 수준: 중간
   - 영향 받는 이유: 알루미늄 압출업체로서 원자재 가격 상승이 영향을 미치지만, 제품의 가격 조정 가능성으로 인해 리스크가 다소 완화될 수 있음.

4. **동남**
   - 리스크 수준: 높음
   - 영향 받는 이유: 알루미늄 주조업체로서, 알루미늄 가격 상승이 직접적인 원가 상승으로 이어질 가능성이 높음.

5. **삼미금속**
   - 리스크 수준: 중간
   - 영향 받는 이유: 금속 단조 제조업체로, 원자재 가격 상승이 영향을 미치지만, 대체 원자재 사용 가능성으로 인해 리스크가 다소 완화될 수 있음.

## 3. 수입 원자재/품목 리스크
- **수입품목**: 캐스팅 얼로이 (HS코드: 7601201000)
- **영향 받는 국가**: 이집트, 사우디아라비아, 바레인 등 중동 국가
- **단가 변동 예상**: 현재 알루미늄 가격이 급등하고 있어, 캐스팅 얼로이의 평균 단가도 상승할 것으로 예상됨.

## 9. 배치 분석: 여러 뉴스 한꺼번에 처리

In [24]:
def batch_analyze(
    news_df: pd.DataFrame,
    n_news: int = 5,
    filter_keyword: str = None,
    score_threshold: float = 0.35
) -> pd.DataFrame:
    """
    여러 뉴스를 배치로 분석하여 결과 DataFrame 반환

    Parameters
    ----------
    news_df         : 뉴스 DataFrame
    n_news          : 분석할 뉴스 수
    filter_keyword  : 제목 필터링 키워드 (None이면 전체)
    score_threshold : 협력사 포함 최소 유사도
    """
    if filter_keyword:
        subset = news_df[news_df["title"].str.contains(filter_keyword, na=False)].head(n_news)
    else:
        subset = news_df.head(n_news)

    results = []
    for _, row in tqdm(subset.iterrows(), total=len(subset), desc="뉴스 분석"):
        # 빠른 검색 결과만 수집 (API 호출 없이)
        partners, imports = retrieve_by_news(
            row["title"], str(row.get("content", ""))
        )
        high_risk_partners = [
            p["meta"]["회사명"] for p in partners if p["score"] >= score_threshold
        ]
        top_countries = list({
            i["meta"]["country_name"] for i in imports if i["score"] >= score_threshold
        })
        top_items = list({
            i["meta"]["item_name"] for i in imports if i["score"] >= score_threshold
        })

        results.append({
            "뉴스 제목": row["title"],
            "발행일": row.get("pub_date", ""),
            "영향 협력사 수": len(high_risk_partners),
            "영향 협력사 목록": ", ".join(high_risk_partners[:5]),
            "관련 수입국": ", ".join(top_countries[:3]),
            "관련 품목": ", ".join(top_items[:3]),
            "최고 유사도(협력사)": partners[0]["score"] if partners else 0,
        })

    return pd.DataFrame(results)


# 키워드 필터링 없이 상위 5개 뉴스 배치 분석
batch_df = batch_analyze(df_news, n_news=5)
print("\n[배치 분석 결과]")
pd.set_option('display.max_colwidth', 40)
print(batch_df.to_string(index=False))

뉴스 분석: 100%|██████████| 5/5 [00:01<00:00,  3.16it/s]


[배치 분석 결과]
                                           뉴스 제목        발행일  영향 협력사 수                                영향 협력사 목록       관련 수입국                         관련 품목  최고 유사도(협력사)
                           로봇은 왜 인간의 모습을 하고 있는가? 2026-03-31        10          레인보우로보틱스, 로보케어, 에브리봇, 큐렉소, 로보티즈                                                0.800443
                [결정의 순간들①] 정주영·정세영의 유산 'HDC 정몽규' 2026-03-31        10 한국파워트레인, 케이비아이동국실업, 한국무브넥스, 현대피앤씨, 평화홀딩스                                                0.410463
   [이슈체크] 이란전쟁 한달 만에 시총 840조 증발…'삼전닉스'만 372조원... 2026-03-31         4           현대차증권, 현대차미소금융재단, 현대캐피탈, 현대커머셜 이집트, 일본, 레바논 백금이나 백금화합물의 것, 기타, 복합구조칩 집적회로     0.444864
[TECH한주] 'K-라이다'의 자존심 에스오에스랩, 피지컬 AI의 '눈'을 넘어... 2026-03-31        10          인터플렉스, 씨티에스, 영진테크, 에스아이티, 에스티아이                                                0.593931
 YTN '저널리즘 책무위' 가동… 노조 &quot;유진 강점기 합리화 명분&quot; 2026-03-30        10            리어코리아, 덴소코리아, 세방, 삼우코리아, 서진산업                                           

## 10. 특정 협력사 중심 역방향 분석

특정 협력사를 입력하면, 그 협력사에 영향을 줄 수 있는 뉴스를 찾습니다.

In [25]:
def find_news_for_partner(company_name: str, top_k: int = 5) -> pd.DataFrame:
    """
    협력사명 → 관련 리스크 뉴스 탐색
    """
    # 협력사 정보 조회
    partner_row = df_partners[df_partners["회사명"].str.contains(company_name, na=False)]
    if partner_row.empty:
        print(f"'{company_name}' 협력사를 찾을 수 없습니다.")
        return pd.DataFrame()

    p = partner_row.iloc[0]
    query = f"{p['회사명']} {p['업종']} {p['비즈니스 요약']}"

    print(f"\n[협력사 정보]")
    print(f"  회사명: {p['회사명']}")
    print(f"  업종: {p['업종']}")
    print(f"  비즈니스: {p['비즈니스 요약']}")
    print()

    # 관련 뉴스 검색
    news_results = retrieve(query, news_index, news_docs, df_news, top_k)

    rows = []
    for r in news_results:
        rows.append({
            "유사도": round(r["score"], 3),
            "뉴스 제목": r["meta"]["title"],
            "발행일": r["meta"].get("pub_date", ""),
        })

    return pd.DataFrame(rows)


# 사용 예시: 협력사명 일부로 검색
result_df = find_news_for_partner("현대트랜시스")
if not result_df.empty:
    print("[관련 뉴스 탐색 결과]")
    pd.set_option('display.max_colwidth', 60)
    print(result_df.to_string(index=False))


[협력사 정보]
  회사명: 현대트랜시스
  업종: 자동차 부품
  비즈니스: 변속기, 액슬, 시트 등 자동차 핵심 파워트레인 및 내장 부품 제조

[관련 뉴스 탐색 결과]
  유사도                               뉴스 제목        발행일
0.720      대동금속, 현대차 실린더헤드 납품… 독보적 기술력 입증 2026-03-29
0.668          현대차·기아, iF 디자인 어워드 32관왕 달성 2026-03-26
0.664   대동금속, 농기계 넘어 볼보·현대차까지… 7만 톤 주물 신화 2026-03-27
0.643 현대차그룹 미래車, ‘현대차 뉴테크+기아 전동화’ 더블엔진 장착 2026-03-30
0.640              폴란드-중동… 기아 군용차 30개국 질주 2026-03-30


In [26]:
# 역방향 분석 후 전체 리스크 보고서 생성
company_to_analyze = "현대트랜시스"  # ← 여기서 협력사명 변경

news_for_partner = find_news_for_partner(company_to_analyze, top_k=3)

if not news_for_partner.empty:
    top_news_title = news_for_partner.iloc[0]["뉴스 제목"]
    top_news_row = df_news[df_news["title"] == top_news_title]

    if not top_news_row.empty:
        top_row = top_news_row.iloc[0]
        partner_report = analyze_risk(
            news_title=top_row["title"],
            news_content=str(top_row.get("content", "")),
            news_date=str(top_row.get("pub_date", "")),
            verbose=True
        )
        print("=" * 70)
        print(f"📊 '{company_to_analyze}' 관련 리스크 보고서")
        print("=" * 70)
        print(partner_report)


[협력사 정보]
  회사명: 현대트랜시스
  업종: 자동차 부품
  비즈니스: 변속기, 액슬, 시트 등 자동차 핵심 파워트레인 및 내장 부품 제조

🔍 뉴스 분석 시작: 대동금속, 현대차 실린더헤드 납품… 독보적 기술력 입증...
   협력사 후보 10개, 수입품목 8개 검색 완료
   OPEN API 호출 중...
📊 '현대트랜시스' 관련 리스크 보고서
## 1. 뉴스 핵심 리스크 요인
1. **대동금속의 독점적 기술력**: 현대차와의 합작 개발로 인해 대동금속이 실린더 헤드의 독점 공급자로 자리잡음.
2. **부품 공급망 집중화**: 특정 협력사에 의존하는 구조로 인해 공급 차질 발생 시 리스크가 증가.
3. **기술적 의존성**: L엔진용 실린더 헤드와 같은 핵심 부품의 기술적 의존성이 높아, 대체 공급처 확보의 어려움.
4. **시장 경쟁 심화**: 대동금속의 성장은 경쟁사에 압박을 가할 수 있으며, 이는 가격 및 공급 안정성에 영향을 미칠 수 있음.

## 2. 영향 받는 협력사 분석
1. **회사명**: 대동금속  
   **리스크 수준**: 낮음  
   **영향 받는 이유**: 현재 현대차에 실린더 헤드를 독점 공급하고 있어, 공급망의 안정성에 기여하고 있음.

2. **회사명**: 케이피에프  
   **리스크 수준**: 중간  
   **영향 받는 이유**: 금속 가공 제품 제조업체로서, 대동금속의 부품 공급에 의존할 가능성이 있으며, 대동금속의 공급 차질이 발생할 경우 영향을 받을 수 있음.

3. **회사명**: 신영정밀  
   **리스크 수준**: 중간  
   **영향 받는 이유**: 자동차용 정밀 부품을 생산하는 업체로, 대동금속의 부품과 연계된 공급망에 위치할 수 있음.

4. **회사명**: 일진공업  
   **리스크 수준**: 중간  
   **영향 받는 이유**: 조향 장치 및 현가 장치 부품을 공급하는 업체로, 대동금속의 부품과의 연관성이 있을 수 있음.

5. **회사명**: 현대트랜시스  
   **리스크 수준**:

## 11. 대화형 인터페이스 (Interactive Mode)

셀을 실행하면 뉴스 제목을 직접 입력하여 실시간 분석할 수 있습니다.

In [27]:
def interactive_rag():
    """
    대화형 RAG 실행 루프
    'quit' 입력 시 종료
    """
    print("" * 2)
    print("="*60)
    print("🚨 HMG 공급망 리스크 탐지 시스템")
    print("="*60)
    print("뉴스 제목을 입력하면 영향 받는 협력사를 분석합니다.")
    print("종료하려면 'quit' 입력\n")

    while True:
        title = input("📰 뉴스 제목: ").strip()
        if title.lower() in ["quit", "exit", "q"]:
            print("종료합니다.")
            break
        if not title:
            continue

        content = input("📄 뉴스 본문 (없으면 Enter): ").strip()

        report = analyze_risk(
            news_title=title,
            news_content=content,
            verbose=True
        )

        print("\n" + "="*60)
        print(report)
        print("="*60 + "\n")


# 대화형 모드 실행 (주석 해제 후 사용)
# interactive_rag()

## 12. 인덱스 저장 & 로드 (재실행 시 속도 향상)

In [28]:
import pickle

def save_indices(save_dir="./rag_indices"):
    """FAISS 인덱스와 메타데이터를 파일로 저장"""
    os.makedirs(save_dir, exist_ok=True)

    faiss.write_index(partner_index, f"{save_dir}/partner.index")
    faiss.write_index(import_index,  f"{save_dir}/import.index")
    faiss.write_index(news_index,    f"{save_dir}/news.index")

    with open(f"{save_dir}/docs_meta.pkl", "wb") as f:
        pickle.dump({
            "partner_docs": partner_docs,
            "import_docs":  import_docs,
            "news_docs":    news_docs,
            "df_partners":  df_partners,
            "df_imp_agg":   df_imp_agg,
            "df_news":      df_news,
        }, f)

    print(f"✅ 인덱스 저장 완료: {save_dir}/")


def load_indices(save_dir="./rag_indices"):
    """저장된 인덱스 로드 (임베딩 재계산 불필요)"""
    global partner_index, import_index, news_index
    global partner_docs, import_docs, news_docs
    global df_partners, df_imp_agg, df_news

    partner_index = faiss.read_index(f"{save_dir}/partner.index")
    import_index  = faiss.read_index(f"{save_dir}/import.index")
    news_index    = faiss.read_index(f"{save_dir}/news.index")

    with open(f"{save_dir}/docs_meta.pkl", "rb") as f:
        data = pickle.load(f)

    partner_docs = data["partner_docs"]
    import_docs  = data["import_docs"]
    news_docs    = data["news_docs"]
    df_partners  = data["df_partners"]
    df_imp_agg   = data["df_imp_agg"]
    df_news      = data["df_news"]

    print(f"✅ 인덱스 로드 완료: {save_dir}/")


# 저장 실행
save_indices()

✅ 인덱스 저장 완료: ./rag_indices/


---
## 📌 사용 가이드

| 목적 | 사용 셀 |
|------|--------|
| 특정 뉴스 리스크 분석 | Section 7 `analyze_risk()` |
| 직접 뉴스 텍스트 입력 | Section 8 커스텀 입력 |
| 여러 뉴스 한번에 분석 | Section 9 `batch_analyze()` |
| 협력사 → 관련 뉴스 탐색 | Section 10 `find_news_for_partner()` |
| 실시간 대화형 입력 | Section 11 `interactive_rag()` |
| 인덱스 재사용 (속도 향상) | Section 12 `load_indices()` |

### 파라미터 튜닝
- `TOP_K_PARTNERS` / `TOP_K_IMPORTS`: 검색 후보 수 조절
- `partner_score_threshold`: 유사도 기준값 조절 (0.3~0.5 권장)
- `EMBEDDING_MODEL`: 다른 한국어 모델로 교체 가능 (`jhgan/ko-sroberta-multitask` 등)


In [29]:
# SYSTEM_PROMPT = """
# 당신은 현대자동차그룹(HMG) 구매/공급망 전문 리스크 분석 AI입니다.
# 주어진 뉴스와 검색된 협력사·수입 품목 데이터를 바탕으로,
# 어떤 협력사가 영향을 받는지 판단하고 그 이유를 체계적으로 정리합니다.

# 분석 시 다음 원칙을 따릅니다:
# 1. 뉴스에서 핵심 리스크 요인(지정학적 이슈, 원자재 가격, 공급 차질 등)을 식별합니다.
# 2. 검색된 협력사 중 실제로 영향을 받을 가능성이 높은 업체를 선별합니다.
# 3. 수입 품목 데이터와 연계하여 원자재/소재 공급 리스크를 분석합니다.
# 4. 리스크 수준을 [높음/중간/낮음]으로 명확히 표시합니다.
# 5. 권고 조치(선제적 단가 협상, 대체 공급선 검토 등)를 제안합니다.
# 6. 근거가 부족한 연결고리는 추측임을 명시합니다.

# [데이터 사용 원칙 - 반드시 준수]
# 1. 모든 답변은 반드시 RAG 검색을 통해 제공된 벡터DB 데이터에 근거해야 합니다.
#    DB에 없는 협력사, 품목, 수치 등을 임의로 생성하거나 추론하여 답변에 포함하지 마십시오.
# 2. 검색 결과에 관련 데이터가 존재하지 않을 경우,
#    "현재 DB에서 관련 정보를 찾을 수 없습니다"라고 명확히 고지한 후 답변을 작성하십시오.
# 3. DB 데이터와 일반 지식을 혼용할 경우, 반드시 출처를 구분하여 표시하십시오.
#    - DB 기반: "[DB]" 태그 사용
#    - 일반 지식 기반: "[일반]" 태그 사용
# """

In [30]:
SYSTEM_PROMPT = """
당신은 현대자동차그룹(HMG) 구매/공급망 전문 리스크 분석 AI입니다.

[절대 원칙 - 위반 금지]
- [RAG 검색 결과]에 언급되지 않은 협력사명, 품목명, 수치, 국가는
  절대로 답변에 포함하지 마십시오.
- 관련 데이터가 없으면 반드시 아래 형식으로만 답변하십시오:
  "현재 DB에 관련 데이터가 없어 답변드리기 어렵습니다."

[분석 원칙]
1. 뉴스에서 핵심 리스크 요인(지정학적 이슈, 원자재 가격, 공급 차질 등)을 식별합니다.
2. [RAG 검색 결과]에 존재하는 협력사만 언급합니다.
3. 수입 품목도 [RAG 검색 결과]에 있는 것만 분석합니다.
4. 리스크 수준을 [높음/중간/낮음]으로 표시합니다.
5. 권고 조치는 DB에 근거한 내용만 제안합니다.
6. DB 기반 내용은 [DB], 뉴스 기반 내용은 [뉴스]로 출처를 표시합니다.
"""

In [31]:
def chat_with_rag(
    user_message: str,
    conversation_history: list = None,
    use_rag: bool = True,          # None이면 자동 판단
    verbose: bool = True
) -> tuple[str, list]:
    """
    RAG 기반 자유 질문 챗봇

    Parameters
    ----------
    user_message          : 사용자 입력 (자유 질문 or 뉴스 분석 요청)
    conversation_history  : 이전 대화 내역 (없으면 새 대화 시작)
    use_rag               : RAG 검색 강제 여부 (None=자동 판단)
    verbose               : 중간 과정 출력 여부

    Returns
    -------
    tuple[str, list] : (AI 응답, 업데이트된 대화 내역)
    """

    if conversation_history is None:
        conversation_history = []

    # ── Step 1: RAG 검색 (항상 실행) ─────────────────────────
    rag_context = ""
    if use_rag:
        partners, imports = retrieve_by_news(user_message, "")
        rag_context = format_retrieved_context(partners, imports)

        if verbose:
            print(f"   협력사 후보 {len(partners)}개, 수입품목 {len(imports)}개 검색 완료")

    # ── Step 2: 메시지 구성 ───────────────────────────────────

    # RAG 결과가 있으면 user 메시지에 context 첨부
    if rag_context:
        augmented_message = f"""{user_message}

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📦 관련 데이터 (RAG 검색 결과)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
{rag_context}"""
    else:
        augmented_message = user_message

    # 대화 히스토리에 현재 메시지 추가
    updated_history = conversation_history + [
        {"role": "user", "content": augmented_message}
    ]

    # ── Step 3: API 호출 ─────────────────────────────────────
    if verbose:
        print("   API 호출 중...")

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            *updated_history            # 전체 대화 히스토리 전달
        ],
        max_tokens=3000,
        temperature=0
    )

    assistant_message = response.choices[0].message.content

    # 히스토리에 AI 응답 추가 (다음 턴에 전달할 용도)
    updated_history.append({"role": "assistant", "content": assistant_message})

    return assistant_message, updated_history


print("✅ 챗봇 함수 정의 완료")

✅ 챗봇 함수 정의 완료


In [43]:
# ── 새 대화 시작 ──────────────────────────────────────────────
history = []

response, history = chat_with_rag(
    "현재 우리 기업의 공급망 관련 문제가 생길만한 공급처가 있을까?",
    conversation_history=history
)
print(response)



   협력사 후보 10개, 수입품목 8개 검색 완료
   API 호출 중...
현재 공급망 관련 리스크 요인은 다음과 같습니다:

1. **협력사: 우림피티에스**
   - 리스크 요인: 기어박스 및 동력 전달 부품 제조업체로, 원자재 가격 변동 및 공급 차질 가능성이 존재.
   - 리스크 수준: 중간
   - 권고 조치: 공급망 다변화 및 원자재 가격 모니터링 강화 [DB].

2. **협력사: 하이록코리아**
   - 리스크 요인: 차량용 정밀 계장 피팅 및 밸브 공급업체로, 공급망 안정성에 대한 우려.
   - 리스크 수준: 중간
   - 권고 조치: 공급업체와의 긴밀한 커뮤니케이션 및 재고 관리 강화 [DB].

3. **협력사: 비에이치**
   - 리스크 요인: 전장 디스플레이용 FPCB 공급업체로, 전자부품 수급 불안정성.
   - 리스크 수준: 중간
   - 권고 조치: 대체 공급처 발굴 및 재고 확보 [DB].

4. **협력사: 이수페타시스**
   - 리스크 요인: 자율주행 서버용 다층 PCB 공급업체로, 기술적 변화에 따른 리스크.
   - 리스크 수준: 중간
   - 권고 조치: 기술 동향 모니터링 및 공급망 유연성 확보 [DB].

5. **협력사: 동양이엔피**
   - 리스크 요인: 전기차 충전기용 파워 모듈 및 전원 공급 장치 제조업체로, 전기차 시장의 변화에 따른 리스크.
   - 리스크 수준: 중간
   - 권고 조치: 시장 변화에 대한 분석 및 공급망 조정 [DB].

6. **협력사: 태원정밀**
   - 리스크 요인: 자동차 인젝터 부품 및 고압 밸브 시트 제조업체로, 부품 수급의 불확실성.
   - 리스크 수준: 중간
   - 권고 조치: 공급망 안정성 확보를 위한 다각적 접근 [DB].

이 외에도 기어박스와 관련된 수입 품목이 여러 국가에서 이루어지고 있어, 해당 품목의 공급망 리스크를 지속적으로 모니터링할 필요가 있습니다.


In [32]:
# ── 새 대화 시작 ──────────────────────────────────────────────
history = []

response, history = chat_with_rag(
    "중국에서 수입하는 품목이 뭐야?",
    conversation_history=history
)
print(response)

   협력사 후보 10개, 수입품목 8개 검색 완료
   API 호출 중...
중국에서 수입하는 품목은 다음과 같습니다:

- 품목: 점화용 와이어링 세트와 그 밖의 와이어링 세트(자동차용ㆍ항공기용ㆍ선박용으로 한정한다)
- 평균단가: 20.19 USD/kg
- 총수입액: 1,605,245,460 천달러 [DB]


In [33]:
# ── 새 대화 시작 ──────────────────────────────────────────────
history = []

response, history = chat_with_rag(
    "철강 관련 협력사 목록 알려줘",
    conversation_history=history
)
print(response)

   협력사 후보 10개, 수입품목 8개 검색 완료
   API 호출 중...
철강 관련 협력사 목록은 다음과 같습니다:

1. 신한산업 - 철강 자재 도매업 (자동차용 특수강 및 일반 철강재 유통 서비스)
2. 경남스틸 - 철강/가공 (포스코 협력사로서 자동차용 냉연 강판 정밀 가공)
3. 삼현철강 - 철강/유통 (산업용 강판 코일 센터 및 철강재 유통)
4. 신한철강 - 철강재 임가공업 (열연 및 냉연 강판의 슬리팅 및 커팅 가공 서비스)
5. 문배철강 - 철강/유통 (열연 강판 및 자동차 구조용 강판 가공)


In [34]:
# ── 새 대화 시작 ──────────────────────────────────────────────
history = []

response, history = chat_with_rag(
    "배터리 공급을 담당하는 협력사는?",
    conversation_history=history
)
print(response)

   협력사 후보 10개, 수입품목 8개 검색 완료
   API 호출 중...
배터리 공급을 담당하는 협력사는 다음과 같습니다:

1. 원익피앤이 - 전기차 배터리 활성화 공정 장비 및 충전기 생산
2. 파워로직스 - 전기차용 배터리 보호 회로 및 배터리 팩 솔루션
3. SK온 - 전기차용 리튬이온 배터리 셀 제조
4. 비츠로셀 - 리튬 일차전지 및 자동차 전장용 전원 솔루션
5. 엘앤에프 - 이차전지용 양극활물질 생산 및 공급
6. 솔루스첨단소재 - 전기차 배터리 핵심 소재 전지박 생산
7. LG에너지솔루션 - 전기차용 파우치형 배터리 셀 공급
8. 에코프로비엠 - 하이니켈 배터리 양극재 전문 제조
9. 엠플러스 - 파우치형 전기차 배터리 조립 공정 자동화 장비 제조
10. 코스모신소재 - 양극재 및 배터리용 이형필름 제조


In [35]:
# ── 새 대화 시작 ──────────────────────────────────────────────
history = []

response, history = chat_with_rag(
    "알루미늄 단가 급등하면 어떤 협력사가 타격받아?",
    conversation_history=history
)
print(response)

   협력사 후보 10개, 수입품목 8개 검색 완료
   API 호출 중...
알루미늄 단가 급등 시 타격을 받을 협력사는 다음과 같습니다:

1. **남선알미늄**: 알루미늄 압출업체로, 자동차용 범퍼 가드 및 알루미늄 외장 부품을 제조하고 있어 원자재 가격 상승에 민감합니다. [뉴스]
   - 리스크 수준: 높음

2. **피제이메탈**: 알루미늄 제련업체로, 자동차 부품용 알루미늄 탈산제 및 합금 원료를 공급합니다. 원자재 가격 상승이 직접적인 영향을 미칠 수 있습니다. [뉴스]
   - 리스크 수준: 높음

3. **삼아알미늄**: 전기차 이차전지용 알루미늄 박을 전문 생산하는 업체로, 알루미늄 가격 상승이 생산 비용에 영향을 줄 수 있습니다. [뉴스]
   - 리스크 수준: 높음

4. **세진메탈**: 자동차용 알루미늄 합금 및 알루미늄 잉곳을 생산하는 업체로, 원자재 가격 상승에 따른 비용 증가가 우려됩니다. [뉴스]
   - 리스크 수준: 높음

5. **삼보산업**: 자동차 엔진 및 변속기 부품용 알루미늄 합금괴를 제조하는 업체로, 알루미늄 가격 상승이 직접적인 영향을 미칠 것입니다. [뉴스]
   - 리스크 수준: 높음

6. **알루코**: 전기차 배터리 팩 케이스 및 자동차 경량화용 알루미늄 압출재를 생산하는 업체로, 원자재 가격 상승이 생산 비용에 영향을 줄 수 있습니다. [뉴스]
   - 리스크 수준: 높음

7. **조일알미늄**: 자동차 열교환기 및 차체용 알루미늄 판재를 공급하는 업체로, 알루미늄 가격 상승이 비용에 영향을 미칠 수 있습니다. [뉴스]
   - 리스크 수준: 높음

8. **대진금속**: 자동차 변속기용 알루미늄 케이스 및 밸브 바디 주물을 제조하는 업체로, 원자재 가격 상승이 우려됩니다. [뉴스]
   - 리스크 수준: 높음

9. **케이에이치바텍**: 전기차용 알루미늄/마그네슘 방열 부품 및 하우징을 제조하는 업체로, 알루미늄 가격 상승이 영향을 미칠 수 있습니다. [뉴스]
   - 리스크 수준: 중간

10

In [36]:
!pip install ragas==0.2.15

In [40]:
# 평가하고 싶은 다양한 시나리오 질문들
test_questions = [
    "현재 우리 기업의 공급망 관련 문제가 생길만한 공급처가 있을까?",
    "우리 기업의 차의 핵심 부품 수주받는 공급처중에 문제가 생긴 공급처가 있어?",
    "엔진 생산을 대체할만한 공급처가 있을까?",
    "철강 관련 협력사 목록 알려줘",
    "중국에서 수입하는 품목이 뭐야?",
    "알루미늄 단가 급등하면 어떤 협력사가 타격받아?",
    "현대자동차가 파트너십을 맺은 기업은?"
]

In [41]:
import pandas as pd
from datasets import Dataset

# 데이터를 담을 리스트
questions = []
answers = []
contexts = []

print("--- [데이터셋 생성 시작] ---")

for q in test_questions:
    print(f"질문 처리 중: {q}")

    # 1. 그래프 정보 추출
    partners, imports = retrieve_by_news(q, "")
    ctx = format_retrieved_context(partners, imports)

    # 2. LLM 답변 생성
    ans, _ = chat_with_rag(q)

    # 3. 리스트에 저장
    questions.append(q)
    answers.append(ans)
    contexts.append([ctx]) # Ragas 형식에 맞게 리스트로 감쌈

# 4. Ragas용 Dataset 객체로 변환
eval_dataset = Dataset.from_dict({
    "question": questions,
    "answer": answers,
    "contexts": contexts
})

print(f"--- [총 {len(test_questions)}개의 샘플 데이터셋 생성 완료] ---")

--- [데이터셋 생성 시작] ---
질문 처리 중: 현재 우리 기업의 공급망 관련 문제가 생길만한 공급처가 있을까?
   협력사 후보 10개, 수입품목 8개 검색 완료
   API 호출 중...
질문 처리 중: 우리 기업의 차의 핵심 부품 수주받는 공급처중에 문제가 생긴 공급처가 있어?
   협력사 후보 10개, 수입품목 8개 검색 완료
   API 호출 중...
질문 처리 중: 엔진 생산을 대체할만한 공급처가 있을까?
   협력사 후보 10개, 수입품목 8개 검색 완료
   API 호출 중...
질문 처리 중: 철강 관련 협력사 목록 알려줘
   협력사 후보 10개, 수입품목 8개 검색 완료
   API 호출 중...
질문 처리 중: 중국에서 수입하는 품목이 뭐야?
   협력사 후보 10개, 수입품목 8개 검색 완료
   API 호출 중...
질문 처리 중: 알루미늄 단가 급등하면 어떤 협력사가 타격받아?
   협력사 후보 10개, 수입품목 8개 검색 완료
   API 호출 중...
질문 처리 중: 현대자동차가 파트너십을 맺은 기업은?
   협력사 후보 10개, 수입품목 8개 검색 완료
   API 호출 중...
--- [총 7개의 샘플 데이터셋 생성 완료] ---


In [42]:
from ragas.metrics import Faithfulness, ResponseRelevancy
from ragas import evaluate
from ragas.run_config import RunConfig
from ragas.llms import LangchainLLMWrapper
from langchain_openai import ChatOpenAI
import os

# Ensure the API key is set as an environment variable for Ragas's internal OpenAIEmbeddings
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

evaluator_llm = LangchainLLMWrapper(
    ChatOpenAI(model="gpt-4o", temperature=0)
)

# Create rate-friendly configuration
rate_friendly_config = RunConfig(
    timeout=300,          # 5 minutes max for operations
    max_retries=15,       # More retries for rate limits
    max_wait=90,          # Longer wait between retries
    max_workers=8,        # Fewer concurrent API calls
    log_tenacity=True     # Log retry attempts
)

import pandas as pd
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import Faithfulness, AnswerRelevancy

# 1. 평가 지표 초기화 (정답이 필요 없는 지표만 엄선)
# Faithfulness: 답변이 컨텍스트(뉴스/그래프)에 충실한가?
# AnswerRelevancy: 답변이 질문의 의도에 적절한가?
faithfulness = Faithfulness(llm=evaluator_llm)
answer_relevancy = AnswerRelevancy(llm=evaluator_llm)

# 에러가 났던 ContextPrecision은 제외합니다.
selected_metrics = [faithfulness, answer_relevancy]

# 2. Ragas 평가 실행
print("\n--- [Ragas Reference-free 평가 시작] ---")
results = evaluate(
    dataset=eval_dataset,
    metrics=selected_metrics,       # 수정된 지표 리스트
    llm=evaluator_llm,              # 평가용 LLM
    run_config=rate_friendly_config # 속도 및 재시도 설정
)

# 3. 결과 시각화 및 데이터프레임 변환
df_eval = results.to_pandas()

# 만약 question이 인덱스로 설정되어 있다면 컬럼으로 변환
if 'question' not in df_eval.columns:
    df_eval = df_eval.reset_index()

print("\n--- [평가 결과 리포트] ---")

# 실제로 데이터프레임에 존재하는 컬럼들만 필터링하여 출력 (에러 방지)
available_cols = [col for col in ['question', 'faithfulness', 'answer_relevancy'] if col in df_eval.columns]
print(df_eval[available_cols])

# 4. 최종 평균 점수 요약
print("\n" + "="*40)
print(f"📊 현대차 SCM RAG 시스템 성능 요약")
print("-"*40)

# 평균 계산 시에도 컬럼 존재 여부 확인
if 'faithfulness' in df_eval.columns:
    print(f"✅ 충실도 (Faithfulness): {df_eval['faithfulness'].mean():.4f}")
if 'answer_relevancy' in df_eval.columns:
    print(f"✅ 답변 관련성 (Answer Relevancy): {df_eval['answer_relevancy'].mean():.4f}")

print("="*40)


--- [Ragas Reference-free 평가 시작] ---


Evaluating:   0%|          | 0/14 [00:00<?, ?it/s]


--- [평가 결과 리포트] ---
   faithfulness  answer_relevancy
0      0.260870          0.805319
1      0.000000          0.000000
2      0.000000          0.000000
3      1.000000          0.795803
4      1.000000          0.822975
5      0.428571          0.808193
6      0.000000          0.000000

📊 현대차 SCM RAG 시스템 성능 요약
----------------------------------------
✅ 충실도 (Faithfulness): 0.3842
✅ 답변 관련성 (Answer Relevancy): 0.4618
